<a href="https://colab.research.google.com/github/AmineHamzaoui/Pdf-to-json-using-Langchain-and-FastAPI/blob/main/Pdf_to_json_using_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install langchain

In [ ]:
!pip install langchain_experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 kB 3.6 MB/s eta 0:00:00


In [9]:
pip install langchain-openai langchain-community


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.5 MB/s eta 0:00:00


In [20]:
pip install langchain-openai

In [30]:
pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.0/298.0 kB 6.0 MB/s eta 0:00:00


In [45]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from typing import List, Dict
from langchain_core.prompts import PromptTemplate
from langchain.chat_models import ChatOpenAI
import json

In [58]:
llm=ChatOpenAI(api_key="YOUR_OPENAI_API_KEY")

In [53]:
class Document(BaseModel):
    title: str = Field(description="The title of the document")
    company_name: str = Field(description="Name of the company")
    company_address: str = Field(description="Company's address")
    contact_email: str = Field(description="Contact email for the company")
    product_name: str = Field(description="Commercial name of the product")
    product_codes: List[Dict[str, str]] = Field(description="List of product codes and their details, where each product is represented as a dictionary with attributes like Code Produit, Code ACL13, Dim/Forme, and Boîte")
    composition: str = Field(description="Composition of the product")
    sterilization_process: str = Field(description="Process used for sterilization")
    storage_conditions: str = Field(description="Conditions for storage and conservation")
    indications: List[str] = Field(description="List of indications for use")
    contraindications: str = Field(description="Known contraindications")
    usage_instructions: str = Field(description="Instructions on how to use the product")

parser = JsonOutputParser(pydantic_object=Document)

def load_pdf(file_path):
    loader = PyPDFLoader(file_path)
    pages = loader.load()
    return pages


prompt = PromptTemplate(
    template="Extract the information as specified and for the dict product_codes don't forget the last two rows.\n{format_instructions}\n{context}\n",
    input_variables=["context"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)




In [54]:
pages = load_pdf("/content/drive/MyDrive/1325411001 FT 2010F ADAPTIC 10 X 10 CM.pdf")

chain = prompt | llm | parser

response = chain.invoke({
    "context": pages
})

In [55]:
print(response)

{'title': 'Fiche technique', 'company_name': '3M FRANCE', 'company_address': '1, Parvis de l’Innovation - CS 20203, 95006 Cergy-Pontoise Cedex France', 'contact_email': 'service-marches-3msante@3m.com', 'product_name': 'ADAPTIC™', 'product_codes': [{'Code Produit': '2012', 'Code ACL': '3401073573953', 'Dim/Forme': '7.6 x 20.3 cm', 'Boîte': '10'}, {'Code Produit': '2015', 'Code ACL': '3401073574264', 'Dim/Forme': '12.7 x 22.9 cm', 'Boîte': '12'}, {'Code Produit': '2010F', 'Code ACL13': '3401099681281', 'Dim/Forme': '10 x 10 cm', 'Boîte': '10'}], 'composition': 'Le pansement non adhérent ADAPTIC™ est un pansement primaire constitué d’un tricot de viscose imprégné d’une émulsion stabilisée de vaseline.', 'sterilization_process': 'Irradiation gamma', 'storage_conditions': 'Conserver en dessous de 40°C', 'indications': ['Brûlures 1er & 2ème degrés', 'Eczéma', 'Abrasions', 'Greffes', 'Plaies chirurgicales', 'Ulcères veineux', 'Ulcères du pied diabétique', 'Lacérations', 'Escarres', 'Interven

In [56]:
with open("/content/drive/MyDrive/json_f.json", "w", encoding="utf-8") as json_file:
    json.dump(response, json_file, ensure_ascii=False, indent=4)